# test


This notebook fetches and displays tabular basketball player statistics from the Basketball Stats Vlaanderen website for a specific player and season.

In [1]:
# Import Required Libraries
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def extract_team_name(cell):
    # Find the team name in the nested <a> tag
    link = cell.find('a')
    if link:
        return link.text.strip()
    return cell.text.strip()

def extract_reeks(cell):
    # Find the reeks name in the nested <a> tag
    link = cell.find('a')
    if link:
        return link.text.strip()
    return cell.text.strip()

def get_player_stats(url):
    response = requests.get(url)
    soup = BeautifulSoup(response.text, 'html.parser')
    rows = soup.find_all('tr', class_='mat-mdc-row')
    
    teams, reeks_list, games, pt1_list, pt2_list, pt3_list, tot_list, avg_list = [], [], [], [], [], [], [], []
    
    for row in rows:
        cells = row.find_all('td', class_='mat-mdc-cell')
        if len(cells) >= 8:
            # Team name
            team_name = extract_team_name(cells[0])
            teams.append(team_name)
            # Reeks
            reeks = extract_reeks(cells[1])
            reeks_list.append(reeks)
            # Stats
            games.append(cells[2].text.strip())
            pt1_list.append(cells[3].text.strip())
            pt2_list.append(cells[4].text.strip())
            pt3_list.append(cells[5].text.strip())
            tot_list.append(cells[6].text.strip())
            avg_list.append(cells[7].text.strip())
    
    df = pd.DataFrame({
        'Team': teams,
        'Reeks': reeks_list,
        'Wed': games,
        'pt1': pt1_list,
        'pt2': pt2_list,
        'pt3': pt3_list,
        'Tot': tot_list,
        'avgPts': avg_list
    })
    # Convert numeric columns
    for col in ['Wed', 'pt1', 'pt2', 'pt3', 'Tot', 'avgPts']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Usage:
url = "https://vblstats.wisseq.eu/speler/BVBL744354"
player_stats_df = get_player_stats(url)
display(player_stats_df)

,Team,Reeks,Wed,pt1,pt2,pt3,Tot,avgPts


In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup
import time

def extract_team_name(cell):
    link = cell.find('a')
    if link:
        return link.text.strip()
    return cell.text.strip()

def extract_reeks(cell):
    link = cell.find('a')
    if link:
        return link.text.strip()
    return cell.text.strip()

def get_player_stats_dynamic(url):
    """
    Use Selenium to fetch the page and parse dynamic content.
    If Selenium times out waiting for the table rows, fall back to the
    static requests-based parser `get_player_stats` (defined in another cell).
    """
    options = Options()
    options.add_argument('--headless')
    # Add common flags to make headless Chrome more stable in different environments
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')

    driver = webdriver.Chrome(options=options)
    try:
        driver.get(url)

        try:
            # Increase wait time and be permissive: presence of any row
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "tr.mat-mdc-row"))
            )
        except TimeoutException:
            # If we timed out, check if there is any content at all; if not, fall back
            soup_tmp = BeautifulSoup(driver.page_source, 'html.parser')
            if not soup_tmp.find_all('tr', class_='mat-mdc-row'):
                driver.quit()
                # Fallback to the requests/BeautifulSoup parser from the other cell
                # get_player_stats is defined in a previous cell and will be used here.
                print("Timed out waiting for dynamic rows. Falling back to static parser (requests).")
                return get_player_stats(url)

        # small pause to let any remaining JS settle (optional)
        time.sleep(0.5)
        soup = BeautifulSoup(driver.page_source, 'html.parser')
    finally:
        try:
            driver.quit()
        except Exception:
            pass

    rows = soup.find_all('tr', class_='mat-mdc-row')
    teams, reeks_list, games, pt1_list, pt2_list, pt3_list, tot_list, avg_list = [], [], [], [], [], [], [], []

    for row in rows:
        cells = row.find_all('td', class_='mat-mdc-cell')
        if len(cells) >= 8:
            teams.append(extract_team_name(cells[0]))
            reeks_list.append(extract_reeks(cells[1]))
            games.append(cells[2].text.strip())
            pt1_list.append(cells[3].text.strip())
            pt2_list.append(cells[4].text.strip())
            pt3_list.append(cells[5].text.strip())
            tot_list.append(cells[6].text.strip())
            avg_list.append(cells[7].text.strip())

    # Build dataframe using pandas already imported in another cell
    import pandas as pd
    df = pd.DataFrame({
        'Team': teams,
        'Reeks': reeks_list,
        'Wed': games,
        'pt1': pt1_list,
        'pt2': pt2_list,
        'pt3': pt3_list,
        'Tot': tot_list,
        'avgPts': avg_list
    })
    for col in ['Wed', 'pt1', 'pt2', 'pt3', 'Tot', 'avgPts']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

# Usage:
url = "https://vblstats.wisseq.eu/speler/BVBL744354"
player_stats_df = get_player_stats_dynamic(url)
display(player_stats_df)

Timed out waiting for dynamic rows. Falling back to static parser (requests).


,Team,Reeks,Wed,pt1,pt2,pt3,Tot,avgPts


In [11]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def get_player_stats_dataframe_selenium(url, driver_path='path/to/chromedriver'):
    """
    Scrapes a basketball player's stats from a VBLstats URL using Selenium to handle
    dynamically loaded content and returns a pandas DataFrame.

    Args:
        url (str): The URL of the player's stats page.
        driver_path (str): The path to your Chrome or other browser's WebDriver executable.

    Returns:
        pandas.DataFrame: A DataFrame containing the player's stats, or an empty
                          DataFrame if the data cannot be found.
    """
    try:
        # Set up the WebDriver
        service = Service(driver_path)
        driver = webdriver.Chrome(service=service)
        
        # Open the URL
        driver.get(url)

        # Wait until the table is present on the page
        wait = WebDriverWait(driver, 10)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, 'table')))

        # Get the page source after JavaScript has rendered the table
        page_source = driver.page_source
        
        # Use pandas to read the table from the updated page source
        tables = pd.read_html(page_source)

        if not tables:
            print("No tables found on the webpage.")
            return pd.DataFrame()
        
        # The first table on the page is the one with the stats
        stats_df = tables[0]
        
        # Clean up by closing the browser
        driver.quit()
        
        return stats_df

    except Exception as e:
        print(f"An error occurred: {e}")
        return pd.DataFrame()

# Example usage
# IMPORTANT: You must download the appropriate WebDriver for your browser (e.g., ChromeDriver)
# and provide its path below.
url = "https://vblstats.wisseq.eu/speler/BVBL744354"
driver_path = "C:\\Users\\StijnHuysman\\OneDrive - mateco cloud\\GITHUB REPOS\\HAANTJES\\chromedriver.exe"  # Replace with the actual path
player_stats_df = get_player_stats_dataframe_selenium(url, driver_path)

if not player_stats_df.empty:
   print(player_stats_df)

An error occurred: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=139.0.7258.139)
Stacktrace:
	GetHandleVerifier [0x0x42ffc3+65331]
	GetHandleVerifier [0x0x430004+65396]
	(No symbol) [0x0x223f63]
	(No symbol) [0x0x202e69]
	(No symbol) [0x0x297c7e]
	(No symbol) [0x0x2b24d9]
	(No symbol) [0x0x2912d6]
	(No symbol) [0x0x260910]
	(No symbol) [0x0x261784]
	GetHandleVerifier [0x0x6738b3+2439203]
	GetHandleVerifier [0x0x66eae2+2419282]
	GetHandleVerifier [0x0x45712a+225434]
	GetHandleVerifier [0x0x446e08+159096]
	GetHandleVerifier [0x0x44dd5d+187597]
	GetHandleVerifier [0x0x437ad8+96840]
	GetHandleVerifier [0x0x437c62+97234]
	GetHandleVerifier [0x0x42277a+9962]
	BaseThreadInitThunk [0x0x765e7ba9+25]
	RtlInitializeExceptionChain [0x0x7732c3ab+107]
	RtlClearBits [0x0x7732c32f+191]

